In [13]:
from langchain_community.document_loaders import PyMuPDFLoader

In [16]:
import os
DATA_PATH = "../data/raw"

def load_books():
    all_docs = []

    for file in os.listdir(DATA_PATH):
        if file.endswith(".pdf"):
            file_path = os.path.join(DATA_PATH, file)

            loader = PyMuPDFLoader(file_path)
            docs = loader.load()

            for doc in docs:
                doc.metadata["source"] = file
                doc.metadata["type"] = "book"

            all_docs.extend(docs)

    return all_docs

In [17]:
docs=load_books()

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = splitter.split_documents(docs)

In [19]:
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content)
    print(chunk.metadata)


--- Chunk 0 ---
Aurélien Géron
Hands-on  
Machine Learning  
 with Scikit-Learn,  
Keras & TensorFlow
Concepts, Tools, and Techniques  
to Build Intelligent Systems
TM
2nd Edition
Updated for  
TensorFlow 2
{'producer': 'calibre (4.8.0) [https://calibre-ebook.com]', 'creator': 'calibre (4.8.0) [https://calibre-ebook.com]', 'creationdate': '2019-10-10T14:06:12+00:00', 'source': 'Hands-On Machine Learning.pdf', 'file_path': '../data/raw\\Hands-On Machine Learning.pdf', 'total_pages': 851, 'format': 'PDF 1.6', 'title': 'Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow', 'author': 'Aurélien Géron', 'subject': '', 'keywords': '', 'moddate': '2020-01-16T12:03:44+00:00', 'trapped': '', 'modDate': "D:20200116120344+00'00'", 'creationDate': 'D:20191010140612Z', 'page': 0, 'type': 'book'}

--- Chunk 1 ---
Aurélien Géron
Hands-On Machine Learning with
Scikit-Learn, Keras, and
TensorFlow
Concepts, Tools, and Techniques to
Build Intelligent Systems
SECOND EDITION
Boston
Farnham
S

In [21]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

C:\Users\dell\AppData\Local\Temp\ipykernel_23548\818555509.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1319.75it/s]


In [22]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embedding_model)

In [24]:
vectorstore.save_local("../vector_db/faiss_index")

In [25]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectorstore = FAISS.load_local(
    "../vector_db/faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1638.86it/s]


In [48]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [49]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

In [50]:
prompt = ChatPromptTemplate.from_template("""
You are an ML expert.

Answer the question using ONLY the context below.

Format:
- Use bullet points
- Keep each point short and clear
- Avoid long paragraphs

If you don't know, say "I don't know".

Context:
{context}

Question:
{question}
""")

In [51]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

llm=ChatOpenAI(model="gpt-4",temperature=0)

In [52]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": lambda x: x
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [58]:
query = "What do you about unsupervised machine learning?"

response = rag_chain.invoke(query)

print(response)

- Unsupervised learning is a type of machine learning where the available data is unlabeled, meaning we have the input features X, but not the labels y.
- It is used in situations where the dataset doesn't have labels, which can make it problematic for many practical applications due to the absence of a solid reference point to judge the quality of the model.
- Visualization algorithms are examples of unsupervised learning algorithms. They take complex and unlabeled data and output a 2D or 3D representation of the data, preserving as much structure as possible.
- Some of the most important unsupervised learning algorithms include Clustering (K-Means, DBSCAN, Hierarchical Cluster Analysis), Anomaly detection and novelty detection (One-class SVM, Isolation Forest), and Visualization and dimensionality reduction (Principal Component Analysis, Kernel PCA, Locally Linear Embedding).
- The book only presents unsupervised learning methods that allow building models that can be evaluated based